### Imports necesarios para el Análisis

In [56]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np

### Conexión a la BBDD


In [57]:
MYSQL_PASSWORD='E1q2u3i4p5o33'
MYSQL_HOST='212.227.148.202'
MYSQL_PORT='3306'
MYSQL_USER='Equipo33'
MYSQL_DATABASE='Equip_33'


# Crear motor de conexión
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

### Carga del dataset de la tabla RRHH

In [58]:
with engine.connect() :
    query = "SELECT * FROM RRHH_CLEAN;"
    df = pd.read_sql(query, engine)

### Análisis Desempeño

Pregunta de negocio:
> ¿Cuál es el perfil sociodemográfico de los empleados de la empresa y cómo podemos utilizar esta información para diseñar políticas de talento más adaptadas a sus necesidades y potenciar su compromiso y desarrollo profesional?

La unidad de análisis es el **empleado**, identificado en el dataset mediante la variable `ID`.

Cada fila del dataset original representa una **solicitud de absentismo**.

Los pasos de análisis serán:

- Agregar los registros originales por `ID`, construyendo una tabla resumen por empleado.
- Definir perfiles o segmentos de empleados a partir de variables sociodemográficas.
- Analizar cómo se relacionan estos perfiles con variables como absentismo, rendimiento, antigüedad, carga de trabajo y características personales.
- Identificar patrones que puedan orientar políticas de talento más adaptadas.
- Proponer posibles líneas de actuación en materia de conciliación, desarrollo profesional, bienestar, prevención del absentismo y compromiso.

### Clasificación de variables

In [59]:
# A nivel de dominio de variables:
variables_sociodemograficas = [
    "Age", "Education_Desc", "Son", "Pet", "Social_Drinker", "Social_Smoker", "Body_Mass_Index", 
]

variables_laborales = [
    "Service_Time", "Transportation_Expense", "Distance_Residence_Work", "Work_load_Average_Day"
]

variables_desempeño = [
    "Hit_Target", "Disciplinary_Failure"
]

variables_absencia = [
    "Reason_Absence_Desc",  "Absenteeism_Hours", "Month_Absence", "Day_Week_Desc", "Season_Desc"
]


df_analisis = df[variables_sociodemograficas + variables_laborales + variables_desempeño + variables_absencia].copy()
df_analisis['ID_Employee'] = df['ID_Employee']

### Definir variables como categóricas y booleanas

In [60]:
variables_categoricas = [
    'Reason_Absence_Desc', 'Month_Absence', 'Day_Week_Desc', 'Season_Desc', 'Education_Desc'
]

variables_booleanas = ['Disciplinary_Failure', 'Social_Drinker', 'Social_Smoker']

variables_discretas = ['Son', 'Pet' ]

variables_continuas = [
    'Transportation_Expense', 'Distance_Residence_Work', 'Work_load_Average_Day', 'Age', 
    'Hit_Target', 'Body_Mass_Index', 'Absenteeism_Hours', 'Service_Time'
]

In [61]:
for col in variables_categoricas:
    df_analisis[col] = df_analisis[col].astype("category")

df_analisis["Education_Desc"] = pd.Categorical(
    df_analisis["Education_Desc"],
    categories=[
        "Educación secundaria",
        "Graduado",
        "Postgrado",
        "Máster o doctorado"
    ],
    ordered=True
)

for col in variables_booleanas:
    df_analisis[col] = df_analisis[col].astype("boolean")

df_analisis.info()

<class 'pandas.DataFrame'>
RangeIndex: 702 entries, 0 to 701
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Age                      702 non-null    int64   
 1   Education_Desc           702 non-null    category
 2   Son                      702 non-null    int64   
 3   Pet                      702 non-null    int64   
 4   Social_Drinker           702 non-null    boolean 
 5   Social_Smoker            702 non-null    boolean 
 6   Body_Mass_Index          702 non-null    int64   
 7   Service_Time             702 non-null    int64   
 8   Transportation_Expense   702 non-null    float64 
 9   Distance_Residence_Work  702 non-null    int64   
 10  Work_load_Average_Day    702 non-null    float64 
 11  Hit_Target               702 non-null    int64   
 12  Disciplinary_Failure     702 non-null    boolean 
 13  Reason_Absence_Desc      702 non-null    category
 14  Absenteeism_Hours    

### Generar el dataset por empleado `ID`

#### Comprobar variabilidad del empleado

In [62]:
variables_a_revisar = variables_sociodemograficas + variables_laborales + variables_desempeño + variables_absencia

variabilidad_empleado = (
    df_analisis
    .groupby("ID_Employee")[variables_a_revisar]
    .nunique()
    .reset_index()
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

variabilidad_empleado

,ID_Employee,Age,Education_Desc,Son,Pet,Social_Drinker,Social_Smoker,Body_Mass_Index,Service_Time,Transportation_Expense,Distance_Residence_Work,Work_load_Average_Day,Hit_Target,Disciplinary_Failure,Reason_Absence_Desc,Absenteeism_Hours,Month_Absence,Day_Week_Desc,Season_Desc
0,1,1,1,1,1,1,1,1,1,1,1,17,11,2,13,7,10,5,4
1,2,1,1,1,1,1,1,1,1,1,1,4,2,2,4,3,4,3,2
2,3,1,1,1,1,1,1,1,1,1,1,30,13,2,14,9,12,5,4
3,5,1,1,1,1,1,1,1,1,1,1,14,9,2,5,5,9,5,4
4,6,1,1,1,1,1,1,1,1,1,1,7,5,1,5,2,5,3,4
5,7,1,1,1,1,1,1,1,1,1,1,5,3,2,4,5,4,4,4
6,8,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
7,9,1,1,1,1,1,1,1,1,1,1,5,4,1,5,6,5,3,4
8,10,1,1,1,1,1,1,1,1,1,1,19,9,1,10,6,10,5,4
9,11,1,1,1,1,1,1,1,1,1,1,24,11,2,11,10,11,5,4


#### Generación dataset de empleado

In [63]:
def moda_unica(s):
    """
    Devuelve el valor más frecuente de una serie.
    - Si hay empate, pandas devuelve varias modas; tomamos la primera para obtener
      un único valor por empleado.
    """
    moda = s.mode(dropna=True)
    return moda.iloc[0]

df_empleado = (
    df_analisis
    .groupby("ID_Employee")
    .agg(
        eventos_absentismo=("Absenteeism_Hours", "count"),
        absentismo_total_horas=("Absenteeism_Hours", "sum"),
        absentismo_media_horas=("Absenteeism_Hours", "mean"),
        absentismo_mediana_horas=("Absenteeism_Hours", "median"),
        absentismo_max_horas=("Absenteeism_Hours", "max"),
        mes_ausencia_mas_frecuente=("Month_Absence", moda_unica),
        dia_ausencia_mas_frecuente=("Day_Week_Desc", moda_unica),
        estacion_ausencia_mas_frecuente=("Season_Desc", moda_unica),

        rendimiento_medio=("Hit_Target", "mean"),
        rendimiento_mediana=("Hit_Target", "median"),
        carga_media=("Work_load_Average_Day", "mean"),
        carga_mediana=("Work_load_Average_Day", "median"),

        tuvo_fallo_disciplinario=("Disciplinary_Failure", "max"),

        edad=("Age", "max"),
        educacion=("Education_Desc", "max"),
        hijos=("Son", "max"),
        mascotas=("Pet", "max"),
        bebedor_social=("Social_Drinker", moda_unica),
        fumador_social=("Social_Smoker", moda_unica),

        imc=("Body_Mass_Index", "median"),
        antiguedad=("Service_Time", "max"),
        distancia_trabajo=("Distance_Residence_Work", "median"),
        gasto_transporte=("Transportation_Expense", "median"),
        motivo_ausencia_mas_frecuente=("Reason_Absence_Desc", moda_unica)
    )
    .reset_index()
)

df_empleado

,ID_Employee,eventos_absentismo,absentismo_total_horas,absentismo_media_horas,absentismo_mediana_horas,absentismo_max_horas,mes_ausencia_mas_frecuente,dia_ausencia_mas_frecuente,estacion_ausencia_mas_frecuente,rendimiento_medio,rendimiento_mediana,carga_media,carga_mediana,tuvo_fallo_disciplinario,edad,educacion,hijos,mascotas,bebedor_social,fumador_social,imc,antiguedad,distancia_trabajo,gasto_transporte,motivo_ausencia_mas_frecuente
0,1,23,121,5.260870,4.0,16,8,Jueves,Verano,95.173913,95.0,262.894478,249.7970,True,37,Postgrado,1,1,False,False,29.0,14,11.0,235.0,Consulta médica
1,2,6,25,4.166667,4.5,8,8,Lunes,Verano,93.333333,92.0,241.597000,218.1035,True,48,Educación secundaria,1,5,False,True,33.0,12,29.0,235.0,Sin ausencia registrada
2,3,97,444,4.577320,3.0,32,2,Lunes,Otoño,94.762887,96.0,264.747216,253.9570,True,38,Educación secundaria,0,0,True,False,31.0,18,51.0,179.0,Consulta dental
3,5,18,102,5.666667,8.0,16,9,Lunes,Primavera,91.722222,93.0,266.741389,265.0170,True,43,Educación secundaria,1,0,True,False,38.0,13,20.0,235.0,Ausencia injustificada
4,6,8,72,9.000000,8.0,16,2,Jueves,Otoño,94.875000,94.5,274.829000,274.7285,False,33,Educación secundaria,2,2,False,False,25.0,13,29.0,189.0,Seguimiento de paciente
5,7,6,30,5.000000,3.0,16,3,Jueves,Invierno,94.666667,95.0,303.210833,313.6420,True,39,Educación secundaria,2,0,True,True,24.0,14,5.0,279.0,Enfermedades del sistema genito-urinario
6,8,1,0,0.000000,0.0,0,9,Martes,Verano,81.000000,81.0,294.217000,294.2170,True,39,Educación secundaria,2,2,True,False,35.0,14,35.0,231.0,Sin ausencia registrada
7,9,8,262,32.750000,8.0,120,3,Martes,Otoño,95.875000,96.5,249.042250,255.3390,False,58,Educación secundaria,2,1,False,False,22.0,16,14.0,228.0,Enfermedades del sistema nervioso
8,10,23,178,7.739130,8.0,40,7,Lunes,Verano,94.000000,93.0,258.764348,253.4650,False,28,Educación secundaria,1,4,True,False,27.0,3,52.0,361.0,Seguimiento de paciente
9,11,40,450,11.250000,8.0,104,8,Miércoles,Verano,93.850000,93.5,271.434300,265.3160,True,33,Educación secundaria,2,1,True,False,30.0,13,36.0,289.0,Lesiones y envenenamientos


In [64]:
variables_discretas_empleado = ["hijos", "mascotas"]
variables_booleanas_empleado = ["bebedor_social", "fumador_social", "tuvo_fallo_disciplinario"]
variables_continuas_empleado = [
    "eventos_absentismo", "absentismo_total_horas", "absentismo_media_horas", "absentismo_mediana_horas",
    "absentismo_max_horas", "rendimiento_medio", "rendimiento_mediana", "carga_media","carga_mediana",
     "edad", "imc", "antiguedad", "distancia_trabajo", "gasto_transporte"
]
variables_categoricas_empleado = [
    "educacion", "mes_ausencia_mas_frecuente", "dia_ausencia_mas_frecuente", "estacion_ausencia_mas_frecuente", 
    "motivo_ausencia_mas_frecuente"
] 

### Dataset de Empleado

In [65]:
print(f"Eventos originales: {len(df)}")
print(f"Empleados en tabla agregada: {len(df_empleado)}")
print(f"IDs únicos originales: {df['ID_Employee'].nunique()}")

Eventos originales: 702
Empleados en tabla agregada: 34
IDs únicos originales: 34


#### Estadisticas del dataset Empleado

In [66]:
df_empleado[variables_continuas_empleado+variables_discretas_empleado].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
eventos_absentismo,34.0,20.65,21.64,1.0,6.00,12.50,28.75,97.00
absentismo_total_horas,34.0,148.32,145.32,0.0,26.25,92.50,251.00,476.00
absentismo_media_horas,34.0,7.31,5.70,0.0,4.59,5.50,7.93,32.75
absentismo_mediana_horas,34.0,4.88,2.66,0.0,3.00,4.00,8.00,8.00
absentismo_max_horas,34.0,37.76,38.66,0.0,8.00,16.00,62.00,120.00
rendimiento_medio,34.0,94.28,2.89,81.0,93.89,94.78,95.67,98.50
rendimiento_mediana,34.0,94.53,3.10,81.0,93.62,95.00,96.00,99.00
carga_media,34.0,273.16,20.09,241.6,261.92,268.36,281.72,328.02
carga_mediana,34.0,268.28,20.92,218.1,255.29,265.02,280.49,313.64
edad,34.0,37.85,7.72,27.0,32.00,37.00,42.50,58.00


In [67]:
df_empleado[variables_categoricas_empleado+variables_booleanas_empleado].describe().T


,count,unique,top,freq
educacion,34,4,Educación secundaria,26
mes_ausencia_mas_frecuente,34,9,3,6
dia_ausencia_mas_frecuente,34,5,Lunes,12
estacion_ausencia_mas_frecuente,34,4,Verano,10
motivo_ausencia_mas_frecuente,34,12,Consulta dental,10
bebedor_social,34,2,True,18
fumador_social,34,2,False,27
tuvo_fallo_disciplinario,34,2,True,20


### Perfiles sociodemográficos

In [68]:
df_empleado["grupo_edad"] = pd.cut(
    df_empleado["edad"],
    bins=[0, 30, 40, 50, np.inf],
    labels=["<=30", "31-40", "41-50", ">50"]
)

df_empleado["grupo_educacion"] = df_empleado["educacion"]

df_empleado["grupo_hijos"] = pd.cut(
    df_empleado["hijos"],
    bins=[-1, 0, 1, np.inf],
    labels=["Sin hijos", "1 hijo", "2 o más hijos"]
)

df_empleado["grupo_mascotas"] = pd.cut(
    df_empleado["mascotas"],
    bins=[-1, 0, 1, 2, np.inf],
    labels=["Sin mascotas", "1 mascota", "2 mascotas", "3 o más mascotas"]
)

df_empleado["grupo_bebedor_social"] = df_empleado["bebedor_social"].map({
    False: "No bebedor",
    True: "Bebedor social"
})

df_empleado["grupo_fumador_social"] = df_empleado["fumador_social"].map({
    False: "No fumador",
    True: "Fumador social"
})

df_empleado["grupo_imc"] = pd.cut(
    df_empleado["imc"],
    bins=[0, 18.5, 24.9, 29.9, np.inf],
    labels=["bajo", "normal", "sobrepeso", "obesidad"]
)

df_empleado["grupo_distancia"] = pd.cut(
    df_empleado["distancia_trabajo"],
    bins=[0, 15, 35, np.inf],
    labels=["cerca", "media", "lejos"]
)

df_empleado["grupo_gasto_transporte"] = pd.qcut(
    df_empleado["gasto_transporte"],
    q=3,
    labels=["bajo", "medio", "alto"],
    duplicates="drop"
)

df_empleado[[
    "ID_Employee", "grupo_edad", "grupo_educacion", "grupo_hijos",
    "grupo_mascotas", "grupo_bebedor_social", "grupo_fumador_social", 
    "grupo_imc", "grupo_distancia", "grupo_gasto_transporte"
]]

,ID_Employee,grupo_edad,grupo_educacion,grupo_hijos,grupo_mascotas,grupo_bebedor_social,grupo_fumador_social,grupo_imc,grupo_distancia,grupo_gasto_transporte
0,1,31-40,Postgrado,1 hijo,1 mascota,No bebedor,No fumador,sobrepeso,cerca,medio
1,2,41-50,Educación secundaria,1 hijo,3 o más mascotas,No bebedor,Fumador social,obesidad,media,medio
2,3,31-40,Educación secundaria,Sin hijos,Sin mascotas,Bebedor social,No fumador,obesidad,lejos,bajo
3,5,41-50,Educación secundaria,1 hijo,Sin mascotas,Bebedor social,No fumador,obesidad,media,medio
4,6,31-40,Educación secundaria,2 o más hijos,2 mascotas,No bebedor,No fumador,sobrepeso,media,bajo
5,7,31-40,Educación secundaria,2 o más hijos,Sin mascotas,Bebedor social,Fumador social,normal,cerca,alto
6,8,31-40,Educación secundaria,2 o más hijos,2 mascotas,Bebedor social,No fumador,obesidad,media,medio
7,9,>50,Educación secundaria,2 o más hijos,1 mascota,No bebedor,No fumador,normal,cerca,medio
8,10,<=30,Educación secundaria,1 hijo,3 o más mascotas,Bebedor social,No fumador,sobrepeso,lejos,alto
9,11,31-40,Educación secundaria,2 o más hijos,1 mascota,Bebedor social,No fumador,obesidad,lejos,alto


#### Empleado Perfil Medio

In [69]:
perfil_medio_numerico = (
    df_empleado[variables_continuas_empleado + variables_discretas_empleado]
    .agg(["mean", "median", "min", "max"])
    .T
    .round(2)
)

perfil_medio_categorico = (
    df_empleado[variables_categoricas_empleado + variables_booleanas_empleado]
    .mode(dropna=True)
    .iloc[0]
    .to_frame(name="valor_mas_frecuente")
)

display(perfil_medio_numerico)
display(perfil_medio_categorico)

perfil_medio = pd.concat(
    [
        perfil_medio_numerico['median'],
        perfil_medio_categorico["valor_mas_frecuente"]
    ],
    axis=0
)

perfil_medio

,mean,median,min,max
eventos_absentismo,20.65,12.50,1.0,97.00
absentismo_total_horas,148.32,92.50,0.0,476.00
absentismo_media_horas,7.31,5.50,0.0,32.75
absentismo_mediana_horas,4.88,4.00,0.0,8.00
absentismo_max_horas,37.76,16.00,0.0,120.00
rendimiento_medio,94.28,94.78,81.0,98.50
rendimiento_mediana,94.53,95.00,81.0,99.00
carga_media,273.16,268.36,241.6,328.02
carga_mediana,268.28,265.02,218.1,313.64
edad,37.85,37.00,27.0,58.00


,valor_mas_frecuente
educacion,Educación secundaria
mes_ausencia_mas_frecuente,3
dia_ausencia_mas_frecuente,Lunes
estacion_ausencia_mas_frecuente,Verano
motivo_ausencia_mas_frecuente,Consulta dental
bebedor_social,True
fumador_social,False
tuvo_fallo_disciplinario,True


eventos_absentismo                                 12.5
absentismo_total_horas                             92.5
absentismo_media_horas                              5.5
absentismo_mediana_horas                            4.0
absentismo_max_horas                               16.0
rendimiento_medio                                 94.78
rendimiento_mediana                                95.0
carga_media                                      268.36
carga_mediana                                    265.02
edad                                               37.0
imc                                                25.0
antiguedad                                         12.5
distancia_trabajo                                  25.5
gasto_transporte                                  235.0
hijos                                               1.0
mascotas                                            0.0
educacion                          Educación secundaria
mes_ausencia_mas_frecuente                      

#### Porcentaje representativo de la plantilla de empleados por segmento

In [70]:
segmentos = [
    "grupo_edad", "grupo_educacion", "grupo_hijos", "grupo_mascotas",
    "grupo_bebedor_social", "grupo_fumador_social", "grupo_imc",
    "grupo_distancia", "grupo_gasto_transporte"
]

for col in segmentos:
    tabla = (
        df_empleado[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="empleados")
    )
    tabla["porcentaje"] = (tabla["empleados"] / len(df_empleado) * 100).round(2)
    print(f"\nSegmento: {col}")
    display(tabla)


Segmento: grupo_edad


,grupo_edad,empleados,porcentaje
0,31-40,17,50.00
1,41-50,10,29.41
2,<=30,6,17.65
3,>50,1,2.94



Segmento: grupo_educacion


,grupo_educacion,empleados,porcentaje
0,Educación secundaria,26,76.47
1,Graduado,4,11.76
2,Postgrado,3,8.82
3,Máster o doctorado,1,2.94



Segmento: grupo_hijos


...

,grupo_imc,empleados,porcentaje
0,normal,13,38.24
1,sobrepeso,13,38.24
2,obesidad,8,23.53
3,bajo,0,0.00



Segmento: grupo_distancia


,grupo_distancia,empleados,porcentaje
0,media,15,44.12
1,cerca,10,29.41
2,lejos,9,26.47



Segmento: grupo_gasto_transporte


,grupo_gasto_transporte,empleados,porcentaje
0,bajo,12,35.29
1,medio,11,32.35
2,alto,11,32.35
